In [4]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [5]:
date = read_table("select * from sc_gold.dim_date")
age_group = read_table("select * from sc_gold.dim_agegroup")
qualification = read_table("select * from sc_gold.dim_qualification")
ut = read_table("select * from sc_gold.dim_underemp_type")

In [10]:
df = read_table("select * from sc_bronze.dosm_underemployment")
df

,year,age_group,underemp_type,qualification,underemp_graduate
0,2016,25 - 34,time,degree,6800.0
1,2016,25 - 34,time,diploma,6200.0
2,2016,25 - 34,skill,degree,132200.0
3,2016,25 - 34,skill,diploma,295600.0
4,2016,35 - 44,time,degree,3000.0
...,...,...,...,...,...
139,2024,≤ 24,skill,diploma,193900.0
140,2024,≥ 45,time,degree,7200.0
141,2024,≥ 45,time,diploma,6600.0
142,2024,≥ 45,skill,degree,94100.0


In [7]:
df["date"] = pd.to_datetime(df["year"], format="%Y")
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    age_group[["age_group", "age_group_id"]],
    on="age_group",
    how="left"
)

df=df.merge(
    ut[["underemp_type", "underemp_type_id"]],
    on="underemp_type",
    how="left"
)

df = df.merge(
    qualification[["qualification", "qualification_id"]],
    on="qualification",
    how="left"
)

df_final = df.drop(columns=["year", "date", "age_group","underemp_type", "qualification"])
id_cols = ["date_id", "age_group_id", "underemp_type_id", "qualification_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [8]:
df_final["ue_id"] = ["UE" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["ue_id"] + [c for c in df_final.columns if c != "ue_id"]]
df_final

,ue_id,date_id,age_group_id,underemp_type_id,qualification_id,underemp_graduate
0,UE0001,DT001,AG002,UT002,Q001,6800.0
1,UE0002,DT001,AG002,UT002,Q002,6200.0
2,UE0003,DT001,AG002,UT001,Q001,132200.0
3,UE0004,DT001,AG002,UT001,Q002,295600.0
4,UE0005,DT001,AG003,UT002,Q001,3000.0
...,...,...,...,...,...,...
139,UE0140,DT033,AG001,UT001,Q002,193900.0
140,UE0141,DT033,AG004,UT002,Q001,7200.0
141,UE0142,DT033,AG004,UT002,Q002,6600.0
142,UE0143,DT033,AG004,UT001,Q001,94100.0


In [9]:
write_table(df_final, "sc_gold", "fact_underemp")

Table sc_gold.fact_underemp written successfully.
